In [9]:
from anthropic import Anthropic
from dotenv import load_dotenv
import os

load_dotenv()

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
STOCK_API_KEY = os.getenv("STOCK_API_KEY")


In [10]:
message = []


In [11]:
def user_message(student_message):
    return message.append({"role": "user", "content": student_message})


In [12]:
def assistant_message(assistant_message):
    return message.append({"role": "assistant", "content": assistant_message})

In [ ]:

from rich.console import Console
from rich.markdown import Markdown


console = Console()
SYSTEM_PROMPT = '''
You are a basic medical assistant responsible for answering questions related to medecine. Everything else is prohibited. You are only allowed to respond to these questions:
1. The causes
2. Prevention
3. How it is treated
4. Overview in Rwanda, East Africa, Africa and the world
5. People being killed every year.

You also have access to a tool for looking up real-time stock market prices for publicly traded companies. If a user asks about a stock or share price, use this tool to answer, for any company or exchange the tool has data for. If there is no tool with asked data, respond politely that you don't have it.
When a user asks with a different irrelevant question that is not strictly about diseases/healthcare or about anything related to the tools, tell them you are not eligible to respond to questions that are irrelevant to the scope.

Respond politely and respond if it's just greetings

'''

def chat_response():
    while True:
        request = input("Ask anything related to healthcare:")
        if request.strip().lower() in ("exit"):
            break

        user_message(request)
        print("Request:",request.upper())
        print("Processing your request...")

        while True:
            with client.messages.stream(model="claude-haiku-4-5", temperature=0.7, max_tokens=1000, system=SYSTEM_PROMPT, messages=message, tools=STOCK_TOOLS) as stream:
                for text in stream.text_stream:
                    console.print(Markdown(text), end="")
                response = stream.get_final_message()

            message.append({"role": "assistant", "content": response.content})

            if response.stop_reason != "tool_use":
                break

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = run_stock_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })
            message.append({"role": "user", "content": tool_results})


In [ ]:
# Real-time stock market data via Alpha Vantage

import requests

ALPHA_VANTAGE_URL = "https://www.alphavantage.co/query"


def alpha_vantage_request(params):
    params = {**params, "apikey": STOCK_API_KEY}
    try:
        response = requests.get(ALPHA_VANTAGE_URL, params=params)
        return response.json()
    except requests.RequestException as e:
        return {"error": str(e)}


def get_stock_quote(symbol):
    data = alpha_vantage_request({"function": "GLOBAL_QUOTE", "symbol": symbol})
    try:
        quote = data["Global Quote"]
        return (
            f"Symbol = {quote['01. symbol']}\n"
            f"Stock price = ${quote['05. price']}\n"
            f"Change = ${quote['09. change']}\n"
            f"Volume = {quote['06. volume']}"
        )
    except KeyError:
        return f"No quote data found for symbol '{symbol}'."


STOCK_TOOLS = [
    {
        "name": "get_stock_quote",
        "description": "Get the latest real-time price quote for a publicly traded company. Use this when the user asks for a current stock or share price.",
        "input_schema": {
            "type": "object",
            "properties": {
                "symbol": {
                    "type": "string",
                    "description": "Stock ticker symbol, e.g. AAPL, MSFT, IBM.",
                }
            },
            "required": ["symbol"],
        },
    },
]


def run_stock_tool(name, tool_input):
    if name == "get_stock_quote":
        return get_stock_quote(tool_input["symbol"])
    return f"Unknown tool '{name}'."


In [ ]:
answer = chat_response()